### Import library

In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import AnalyticValue as AV
from matplotlib import pyplot as plt
from scipy.stats import linregress
warnings.filterwarnings(action="ignore")

In [2]:
if not os.path.exists("_Figure"): os.mkdir("_Figure")
if not os.path.exists("_Figure/CGF/"): os.mkdir("_Figure/CGF/")
# if not os.path.exists("_Figure/LDF/"): os.mkdir("_Figure/LDF/")
if not os.path.exists("_Figure/Work/"): os.mkdir("_Figure/Work/")
if not os.path.exists("_Figure/Error/"): os.mkdir("_Figure/Error/")
if not os.path.exists("_Figure/Dynamics/"): os.mkdir("_Figure/Dynamics/")

In [3]:
lmbdas = [-0.1,-0.2,-0.3,-0.4,-0.5,-0.6,-0.7,-0.8,-0.9,-1.0,
          -1.1,-1.2,-1.3,-1.4,-1.5,-1.6,-1.7,-1.8,-1.9,-2.0]
epochs = 50000; L = 1.0
μ = 1.0; τ = 1.0; K = 10.0; Da = 2.0; kbT = 0.5

In [4]:
def f2s(val:float) -> str:
    s = format(val, ".0e")
    s = re.sub(r'e([-+]?)0*(\d+)', r'e\1\2', s)
    return s

### Loss dynamics

In [5]:
def ProcessTrain(what,N,n_hid,n_block,lmbdas,lrp,lrd):
    data_for_all_lmbda = []
    for lmbda in lmbdas:
        data = pd.read_pickle(os.getcwd()+"/data/Equivariant_" + what + "_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl".format(N,n_hid,n_block,lmbda,f2s(lrp),f2s(lrd)))
        data_for_all_lmbda.append(np.array(data))
    return data_for_all_lmbda

def ProcessParam(N,n_hid,n_block,lmbda,lrp,lrd):
    data = pd.read_pickle(os.getcwd()+"/data/Equivariant_PARM_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl".format(N,n_hid,n_block,lmbda,f2s(lrp),f2s(lrd)))
    return np.stack(data)

In [6]:
def PlotLoss(N,n_hid,n_block,lmbdas,lrp,lrd):
    if not os.path.exists('_Figure/Dynamics/N={0}/'.format(N)): os.mkdir('_Figure/Dynamics/N={0}/'.format(N))
    if not os.path.exists('_Figure/Dynamics/N={0}/Loss/'.format(N)): os.mkdir('_Figure/Dynamics/N={0}/Loss/'.format(N))
    true = AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas))
    # load data
    data = ProcessTrain('SCGF',N,n_hid,n_block,lmbdas,lrp,lrd)
    for n in range(len(lmbdas)):
        plt.figure(dpi=400)
        # plot
        x = +np.array([e for e in range(epochs)])
        y = -np.array(data[n])
        plt.ylim(-1.,2.)
        plt.yticks(np.arange(-1.,2.,.1))
        plt.plot(x,y,marker='o',markersize=3,c='blue')
        # 꾸미기
        plt.grid()
        plt.axhline(y=-true[n],c='black')
        plt.xlabel('Epoch'); plt.ylabel('Loss')
        if not os.path.exists('_Figure/Dynamics/N={0}/Loss/lrp={1}_lrd={2}/'.format(N,f2s(lrp),f2s(lrd))):  os.mkdir('_Figure/Dynamics/N={0}/Loss/lrp={1}_lrd={2}/'.format(N,f2s(lrp),f2s(lrd)))
        plt.savefig("_Figure/Dynamics/N={0}/Loss/lrp={1}_lrd={2}/λ={3}_B={4}_H={5}.png".format(N,f2s(lrp),f2s(lrd),lmbdas[n],n_block,n_hid),bbox_inches='tight');
        plt.close();
        plt.clf();
        
N = 21; n_block = 1; n_hid = 25
lrp = 1e-3; lrd = 1e-6
PlotLoss(N,n_hid,n_block,lmbdas,lrp,lrd)

<Figure size 640x480 with 0 Axes>

In [7]:
def PlotParams(N,n_hid,n_block,lmbda,lrp,lrd):
    if not os.path.exists('_Figure/Dynamics/N={0}/'.format(N)): os.mkdir('_Figure/Dynamics/N={0}/'.format(N))
    if not os.path.exists('_Figure/Dynamics/N={0}/Gamma/'.format(N)): os.mkdir('_Figure/Dynamics/N={0}/Gamma/'.format(N))
    for lmbda in lmbdas:
        # load data
        γ = ProcessParam(N,n_hid,n_block,lmbda,lrp,lrd)
        plt.figure(dpi=400)
        plt.plot(γ)
        plt.grid()
        plt.xlabel('epoch'); plt.ylabel('$\\gamma$')
        plt.axhline(y=0.0,c='black')
        plt.ylim(-1.1,+1.1)
        plt.yticks(np.arange(-1.1,1.1,0.1))
        labels = [f"Layer = {i}th" for i in range(n_block+2)]
        plt.legend(labels,bbox_to_anchor=(1., 1.), loc='upper right')
        if not os.path.exists('_Figure/Dynamics/N={0}/Gamma/lrp={1}_lrd={2}/'.format(N,f2s(lrp),f2s(lrd))): os.mkdir('_Figure/Dynamics/N={0}/Gamma/lrp={1}_lrd={2}/'.format(N,f2s(lrp),f2s(lrd)))
        plt.savefig('_Figure/Dynamics/N={0}/Gamma/lrp={1}_lrd={2}/λ={3}_B={4}.png'.format(N,f2s(lrp),f2s(lrd),lmbda,n_block),bbox_inches='tight');
        plt.close();
        plt.clf();
 
N = 21; n_block = 1; n_hid = 25
lrp = 1e-3; lrd = 1e-6
PlotParams(N,n_hid,n_block,lmbdas,lrp,lrd)

<Figure size 640x480 with 0 Axes>

### CGF

In [6]:
def PlotCGF(N,n_hid,n_blocks,lmbdas,lrp,lrd):
    if not os.path.exists("_Figure/CGF/N={0}/".format(N)): os.mkdir("_Figure/CGF/N={0}/".format(N)) 
    if not os.path.exists("_Figure/CGF/N={0}/CGF/".format(N)): os.mkdir("_Figure/CGF/N={0}/CGF/".format(N))
    # true value
    ACGF = AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas))
    # plot
    plt.figure(dpi=400)
    for n_block in n_blocks:
        # load data
        mean = []; std = []
        for n in range(len(lmbdas)):
            data = -pd.read_pickle("result/Equivariant_LSCGF_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lrp),f2s(lrd)))
            # compute mean and std 
            std.append(np.array(data).std())
            mean.append(np.array(data).mean())
        plt.errorbar(lmbdas, mean, std, capsize=3, fmt='o', ls='-', label="B={0}".format(n_block))
    # 꾸미기
    plt.plot(np.linspace(-2.0,0.0,100000), AV.CGF(N,τ,μ,K,Da,kbT,np.linspace(-2.0,0.0,100000)),c='black')
    plt.scatter(lmbdas,ACGF,marker='o',c='black')
    plt.grid()
    plt.legend()
    plt.xlabel("λ")
    plt.ylabel("ψ(λ)")
    plt.ylim(int(2*min(ACGF)), 0.1)
    plt.yticks(np.arange(int(2*min(ACGF)),0.1,0.1))
    plt.savefig("_Figure/CGF/N={0}/CGF/H={1}_lrp={2}.png".format(N,n_hid,f2s(lrp)),bbox_inches='tight');
    plt.close();
    plt.clf();
    
N = 11
n_blocks = [1,3]; n_hid = 15
lrp = 1e-3; lrd = 1e-6

PlotCGF(N,n_hid,n_blocks,lmbdas,lrp,lrd)

<Figure size 640x480 with 0 Axes>

In [10]:
def PlotCGFdiff(N,n_hid,n_block,lmbdas,lrp,lrd):
    if not os.path.exists("_Figure/CGF/N={0}/".format(N)): os.mkdir("_Figure/CGF/N={0}/".format(N)) 
    if not os.path.exists("_Figure/CGF/N={0}/Difference/".format(N)): os.mkdir("_Figure/CGF/N={0}/Difference/".format(N))
    # true value
    ACGF = AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas))
    # plot
    plt.figure(dpi=400)
    mean = []; std = []
    for n in range(len(lmbdas)):
        data = -pd.read_pickle("result/Equivariant_LSCGF_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lrp),f2s(lrd)))
        # compute mean difference and its std
        std.append(abs(data-ACGF[n]).std())
        mean.append(abs(data-ACGF[n]).mean())
    plt.errorbar(lmbdas,mean,std,capsize=3,fmt="o",ls="-",label="L = {0}".format(n_block))
    # 꾸미기
    plt.ylim(-0.1,1.)
    plt.axhline(y=0,c='black')
    plt.yticks(np.arange(-0.1,1.,0.1))
    plt.legend()
    plt.grid()
    plt.xlabel('λ'); plt.ylabel("$<|\hat{ψ}(λ) - ψ(λ)|>$")
    plt.savefig("_Figure/CGF/N={0}/Difference/H={1}_lrp={2}.png".format(N,n_hid,f2s(lrp)),bbox_inches='tight')
    plt.close()
    plt.clf()
  
N = 21; n_block = 1; n_hid = 25
lrp = 1e-3; lrd = 1e-6

PlotCGFdiff(N,n_hid,n_block,lmbdas,lrp,lrd)

<Figure size 640x480 with 0 Axes>

### Observable

In [41]:
def PlotAW(N,n_hid,n_block,lmbdas,lrp,lrds):
    if not os.path.exists("_Figure/Work/N={0}/".format(N)): os.mkdir("_Figure/Work/N={0}/".format(N)) 
    if not os.path.exists("_Figure/Work/N={0}/Work/".format(N)): os.mkdir("_Figure/Work/N={0}/Work/".format(N))
    # true
    AWT = 0; left_list = []; right_list = []
    for k in range(N): left,right = AV.PrimaryDomain(k,N,τ,μ,K,Da,kbT); left_list.append(left); right_list.append(right)
    Almbdas = np.linspace(max(left_list),min(right_list),1000000)
    AWT = AV.AW(N,τ,μ,K,Da,kbT,Almbdas)
    # plot
    plt.figure(dpi=400)
    plt.plot(Almbdas,AWT,c="black")
    plt.scatter(np.array([0.0, -0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0]),
                AV.AW(N,τ,μ,K,Da,kbT,np.array([0.0, -0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0])), c="black")
    # load data
    for lrd in lrds:
        mean = []; std = []
        for n in range(len(lmbdas)):
            data = pd.read_pickle('result/Equivariant_LWORK_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl'.format(N,n_hid,n_block,lmbdas[n],f2s(lrp),f2s(lrd)))
            # compute mean and std
            std.append(np.array(data).std())
            mean.append(np.array(data).mean())
        plt.errorbar(lmbdas,mean,std,capsize=3,fmt="o",ls="-",label="lrp = {0}, lrd = {1}".format(f2s(lrp),f2s(lrd)))
    # 꾸미기
    plt.xlabel("λ"); plt.ylabel("$<\hat{w}(λ)>$")
    plt.xlim(-2.1,0.1)
    plt.grid()
    plt.legend()
    plt.ylim(1.1*min(AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas))),-1.1*min(AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas))))
    plt.savefig("_Figure/Work/N={0}/Work/H={1}_lrp={2}.png".format(N,n_hid,f2s(lrp)),bbox_inches='tight');
    plt.close();
    plt.clf();

N = 11; lrps = [1e-2, 1e-3]
n_block = 1; n_hids = [4]
lrds = [1e-3,1e-4,1e-5,1e-6]

for n_hid in n_hids:
    for lrp in lrps:
        PlotAW(N,n_hid,n_block,lmbdas,lrp,lrds)

<Figure size 640x480 with 0 Axes>

In [ ]:
def PlotAWdiff(N,n_hid,n_block,lmbdas,lrp,lrd):
    if not os.path.exists("_Figure/Work/N={0}/".format(N)): os.mkdir("_Figure/Work/N={0}/".format(N)) 
    if not os.path.exists("_Figure/Work/N={0}/Difference/".format(N)): os.mkdir("_Figure/Work/N={0}/Difference/".format(N))
    # true
    AWT = AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas)) 
    # plot
    plt.figure(dpi=400)
    mean = []; std = []
    for n in range(len(lmbdas)):
        data = pd.read_pickle('result/Equivariant_LWORK_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl'.format(N,n_hid,n_block,lmbdas[n],f2s(lrp),f2s(lrd)))
        std.append(abs(np.array(data)-AWT[n]).std())
        mean.append(abs(np.array(data)-AWT[n]).mean())
    plt.errorbar(lmbdas,mean,std,capsize=3,fmt="o",ls="-",label="B = {0}".format(n_block))
    # 꾸미기
    plt.legend()
    plt.axhline(0,color="black")
    plt.ylim(-0.1,1.)
    plt.yticks(np.arange(-0.1,1.0,0.1))
    plt.grid()
    plt.xlabel("λ"); plt.ylabel("$< |\hat{w}(λ) - w(λ)| >$")
    plt.savefig("_Figure/Work/N={0}/Difference/H={1}_lrp={2}.png".format(N,n_hid,f2s(lrp)),bbox_inches='tight');
    plt.close()
    plt.clf()

N = 21; lrp = 1e-3; lrd = 1e-6
n_block = 1; n_hid = 25

PlotAWdiff(N,n_hid,n_blocks,lmbdas,lrp,lrd)

<Figure size 640x480 with 0 Axes>

In [21]:
def PlotSuperlongAW(N,n_hid,n_block,lmbdas,lrp,lrd):
    # 엄청나게 긴 경로를 만들어서 측정한 active work (입자별)이 학습이 진행됨에 따라서 어떻게 변하는지 (한 line으로 수렴하는지 아닌지) 확인한다
    if not os.path.exists("_Figure/Work/N={0}/".format(N)): os.mkdir("_Figure/Work/N={0}/".format(N)) 
    if not os.path.exists("_Figure/Work/N={0}/SuperLong/".format(N)): os.mkdir("_Figure/Work/N={0}/SuperLong/".format(N))
    # true
    mid_epoch = 1000
    Epochs = [mid_epoch*e for e in range(epochs//mid_epoch+1)] # mid_epoch간격으로 equivariance measure 측정
    AWT = AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas)) / N # 입자의 평균 active work의 true 값
    # plot
    for n in range(len(lmbdas)):
        data = pd.read_pickle("result/Equivariant_EPWORK_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lrp),f2s(lrd)))
        plt.figure(dpi=400) # data.shape = (len(epochs), N)
        for p in range(N):
            plt.plot(Epochs, data[:,p])
        plt.axhline(y=AWT[n],c='black')
        plt.xlabel("Epoch"); plt.ylabel("Active work")
        if not os.path.exists("_Figure/Work/N={0}/SuperLong/lrp={1}_lrd={2}/".format(N,f2s(lrp),f2s(lrd))): os.mkdir("_Figure/Work/N={0}/SuperLong/lrp={1}_lrd={2}/".format(N,f2s(lrp),f2s(lrd)))
        plt.ylim(ymin=AWT[n]*0.5,ymax=AWT[n]*1.5)
        plt.grid()
        plt.savefig("_Figure/Work/N={0}/SuperLong/lrp={1}_lrd={2}/λ={3}_B={4}.png".format(N,f2s(lrp),f2s(lrd),lmbdas[n],n_hid))
        plt.close()
        plt.clf()


N = 11; lrps = [1e-3]
n_block = 1; n_hids = [1,2,3]
lrds = [1e-4,1e-5]

for n_hid in n_hids:
    for lrp in lrps:
        for lrd in lrds:
            PlotSuperlongAW(N,n_hid,n_block,lmbdas,lrp,lrd)

<Figure size 640x480 with 0 Axes>

### Permutation error

In [11]:
def PlotError(N,n_hid,n_block,lmbdas,lrp,lrd):
    epochs = [1000*e for e in range(51)]
    # Relative error
    if not os.path.exists("_Figure/Error/N={0}/".format(N)): os.mkdir("_Figure/Error/N={0}/".format(N))
    for n in range(len(lmbdas)):
        if not os.path.exists("_Figure/Error/N={0}/λ={1}/".format(N,lmbdas[n])): os.mkdir("_Figure/Error/N={0}/λ={1}/".format(N,lmbdas[n])) 
        plt.figure(dpi=400)
        # load data
        data = pd.read_pickle("result/Equivariant_ERROR_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lrp),f2s(lrd)))
        # process data
        pr = []; pa = []
        for e in range(len(epochs)): pr.append(data[e][0]); pa.append(data[e][1])
        # plot
        plt.plot(epochs, pr, alpha=0.7, marker='o', markersize=3, c='red')
        plt.plot(epochs, pa, alpha=0.7, marker='o', markersize=3, c='blue')
        # 꾸미기
        plt.yscale('log'); plt.xlabel('Epoch')
        plt.grid(True, which='both', linestyle='-', linewidth=0.5)
        plt.minorticks_on()
        plt.savefig("_Figure/Error/N={0}/λ={1}/B={2}_lrp={3}_lrd={4}.png".format(N,lmbdas[n],n_block,f2s(lrp),f2s(lrd)),bbox_inches='tight')
        plt.close()
        plt.clf()

N = 11; lrp = 1e-3; lrd = 1e-6
n_block = [1,3]; n_hid = 15

for n_block in n_blocks:
    PlotError(N,n_hid,n_block,lmbdas,lrp,lrd)

<Figure size 640x480 with 0 Axes>

### Fourier

In [44]:
long_models = 1000
# Fourier active work 값을 처리한다.
def ProcessFourier(k,N,n_hid,n_block,lmbdas,lrp,lrd):
    Data = {n: [] for n in range(len(lmbdas))}
    for n in range(len(lmbdas)):
        data = pd.read_pickle("./result/Equivariant_LFORK_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lrp),f2s(lrd)))
        for e in range(long_models):
            Data[n].append(data[e][k])
    return Data    

# k를 나누지 않고 그냥 lmbda에 따라서 묶는다.
def ProcessFourier_fixed(N,n_hid,n_block,lmbdas,lrp,lrd):
    Data = {n: [] for n in range(len(lmbdas))}
    for n in range(len(lmbdas)):
        data = pd.read_pickle("./result/Equivariant_LFORK_N={0}_H={1}_B={2}_λ={3}_lrp={4}_lrd={5}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lrp),f2s(lrd)))
        for e in range(long_models):
            Data[n].append(data[e])
    return Data

In [ ]:
# Plot Fourier work for given λ
def PlotFourier_for_k(N, n_hid, n_block, lmbdas, lrp, lrd):
    if not os.path.exists('_Figure/Work/Fourier/'): os.mkdir('_Figure/Work/Fourier/')
    if not os.path.exists('_Figure/Work/Fourier/N={0}/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/N={0}/Real/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/Real/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/N={0}/Imaginary/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/Imaginary/'.format(N))
    # load data
    data = ProcessFourier_fixed(N,n_hid,n_block,lmbdas,lrp,lrd)
    for n in range(len(lmbdas)):
        list_of_ks = data[n]
        awks = []; std = []
        awks_imag = []; imag_std = []
        for k in range(N):
            for_epoch_real = []
            for_epoch_imag = []
            for e in range(long_models):
                for_epoch_real.append(list_of_ks[e][k].real)
                for_epoch_imag.append(list_of_ks[e][k].imag)
            awks.append(np.array(for_epoch_real).mean())
            std.append(np.array(for_epoch_real).std())
            awks_imag.append(np.array(for_epoch_imag).mean())
            imag_std.append(np.array(for_epoch_imag).std())
        
        # Real value
        if not os.path.exists('_Figure/Work/Fourier/N={0}/λ={1}/'.format(N,lmbdas[n])): os.mkdir('_Figure/Work/Fourier/N={0}/λ={1}/'.format(N,lmbdas[n]))
        plt.figure(dpi=400)
        plt.grid()
        plt.errorbar(np.array([k for k in range(N)]), awks, std, c='#1f77b4', capsize=3, fmt='o', ls="-")
        plt.plot([k for k in range(N)], [AV.AW_k(k,N,τ,μ,K,Da,kbT,lmbdas[n]) for k in range(N)], c='black', markersize=5, marker='o')
        plt.xlabel("$k$"); plt.title("$\\lambda$ = {0}".format(lmbdas[n]))
        plt.savefig('_Figure/Work/Fourier/N={0}/λ={1}/Real_H={4}_lrp={2}_lrd={3}.png'.format(N,lmbdas[n],float_to_sci_str(lrp),float_to_sci_str(lrd),n_hid))
        plt.close()
        plt.clf()

        plt.figure(dpi=400)
        plt.grid()
        plt.errorbar(np.array([k for k in range(N)]), awks_imag, imag_std, c='#ff7f0e', capsize=3, fmt='o', ls="-")
        plt.plot([k for k in range(N)], [0.0 for k in range(N)], c='black', markersize=5, marker='o')
        plt.xlabel("$k$")
        plt.savefig('_Figure/Work/Fourier/N={0}/λ={1}/Imaginary_H={4}_lrp={2}_lrd={3}.png'.format(N,lmbdas[n],float_to_sci_str(lrp),float_to_sci_str(lrd),n_hid))
        plt.close()
        plt.clf()

N = 11; lrp = 1e-2
n_block = 0; n_hids = [4, 5]
lrds = [1e-1,1e-2,1e-3,1e-4,1e-5,1e-6]

for n_hid in n_hids:
    Plotdiff_long(N,n_hid,n_block,lmbdas,lrp,lrds)

<Figure size 640x480 with 0 Axes>

In [ ]:
# Plot Fourier work for given λ
def PlotFourier_for_k_strict(N, n_hid, n_block, lmbdas):
    if not os.path.exists('_Figure/Work/Fourier/'): os.mkdir('_Figure/Work/Fourier/')
    if not os.path.exists('_Figure/Work/Fourier/N={0}/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/N={0}/Real/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/Real/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/N={0}/Imaginary/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/Imaginary/'.format(N))
    # load data
    data = ProcessFourierL_fixed(N,n_hid,n_block,lmbdas)
    for n in range(len(lmbdas)):
        list_of_ks = data[n]
        awks = []; std = []
        awks_imag = []; imag_std = []
        for k in range(N):
            for_epoch_real = []
            for_epoch_imag = []
            for e in range(long_models):
                for_epoch_real.append(list_of_ks[e][k].real)
                for_epoch_imag.append(list_of_ks[e][k].imag)
            awks.append(np.array(for_epoch_real).mean())
            std.append(np.array(for_epoch_real).std())
            awks_imag.append(np.array(for_epoch_imag).mean())
            imag_std.append(np.array(for_epoch_imag).std())
        
        # Real value
        if not os.path.exists('_Figure/Work/Fourier/N={0}/λ={1}/'.format(N,lmbdas[n])): os.mkdir('_Figure/Work/Fourier/N={0}/λ={1}/'.format(N,lmbdas[n]))
        plt.figure(dpi=400)
        plt.grid()
        plt.errorbar(np.array([k for k in range(N)]), awks, std, c='#1f77b4', capsize=3, fmt='o', ls="-")
        plt.plot([k for k in range(N)], [AV.AW_k(k,N,τ,μ,K,Da,kbT,lmbdas[n]) for k in range(N)], c='black', markersize=5, marker='o')
        plt.xlabel("$k$"); plt.title("$\\lambda$ = {0}".format(lmbdas[n]))
        plt.savefig('_Figure/Work/Fourier/N={0}/λ={1}/Real.png'.format(N,lmbdas[n]))
        plt.close()
        plt.clf()

        plt.figure(dpi=400)
        plt.grid()
        plt.errorbar(np.array([k for k in range(N)]), awks_imag, imag_std, c='#ff7f0e', capsize=3, fmt='o', ls="-")
        plt.plot([k for k in range(N)], [0.0 for k in range(N)], c='black', markersize=5, marker='o')
        plt.xlabel("$k$")
        plt.savefig('_Figure/Work/Fourier/N={0}/λ={1}/Imaginary.png'.format(N,lmbdas[n]))
        plt.close()
        plt.clf()

N = 11
n_hid = 5; n_block = 1
PlotFourier_for_k(N, n_hid, n_block, lmbdas)

In [ ]:
# For given λ, compute the error
def PlotFourier_for_k_diff(N, n_hid, n_block, lmbdas):
    if not os.path.exists('_Figure/Work/Fourier/'): os.mkdir('_Figure/Work/Fourier/')
    if not os.path.exists('_Figure/Work/Fourier/Diff/'): os.mkdir('_Figure/Work/Fourier/Diff/')
    if not os.path.exists('_Figure/Work/Fourier/Diff/N={0}/'.format(N)): os.mkdir('_Figure/Work/Fourier/Diff/N={0}/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/Diff/N={0}/Real/'.format(N)): os.mkdir('_Figure/Work/Fourier/Diff/N={0}/Real/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/Diff/N={0}/Imaginary/'.format(N)): os.mkdir('_Figure/Work/Fourier/Diff/N={0}/Imaginary/'.format(N))
    # load data
    data = ProcessFourierL_fixed(N,n_hid,n_block,lmbdas)
    for n in range(len(lmbdas)):
        list_of_ks = data[n]
        awks = []; std = []
        awks_imag = []; imag_std = []
        for k in range(N):
            for_epoch_real = []
            for_epoch_imag = []
            for e in range(long_models):
                for_epoch_real.append(list_of_ks[e][k].real)
                for_epoch_imag.append(list_of_ks[e][k].imag)
            awks.append(abs(np.array(for_epoch_real - AV.AW_k(k,N,τ,μ,K,Da,kbT,lmbdas[n]))).mean())
            std.append(abs(np.array(for_epoch_real- AV.AW_k(k,N,τ,μ,K,Da,kbT,lmbdas[n]))).std())
            awks_imag.append(abs(np.array(for_epoch_imag) - 0.0).mean())
            imag_std.append(abs(np.array(for_epoch_imag) - 0.0).std())
        
        # Real value
        if not os.path.exists('_Figure/Work/Fourier/Diff/N={0}/λ={1}/'.format(N,lmbdas[n])): os.mkdir('_Figure/Work/Fourier/Diff/N={0}/λ={1}/'.format(N,lmbdas[n]))
        plt.figure(dpi=400)
        plt.grid()
        plt.errorbar(np.array([k for k in range(N)]), awks, std, c='#1f77b4', capsize=3, fmt='o', ls="-")
        plt.plot([k for k in range(N)], [AV.AW_k(k,N,τ,μ,K,Da,kbT,lmbdas[n]) for k in range(N)], c='black', markersize=5, marker='o')
        plt.xlabel("$k$"); plt.title("$\\lambda$ = {0}".format(lmbdas[n]))
        plt.savefig('_Figure/Work/Fourier/Diff/N={0}/λ={1}/Long_Real.png'.format(N,lmbdas[n]))
        plt.close()
        plt.clf()

        plt.figure(dpi=400)
        plt.grid()
        plt.errorbar(np.array([k for k in range(N)]), awks_imag, imag_std, c='#ff7f0e', capsize=3, fmt='o', ls="-")
        plt.plot([k for k in range(N)], [0.0 for k in range(N)], c='black', markersize=5, marker='o')
        plt.xlabel("$k$")
        plt.savefig('_Figure/Work/Fourier/Diff/N={0}/λ={1}/Long_Imaginary.png'.format(N,lmbdas[n]))
        plt.close()
        plt.clf()

N = 11
n_hid = 5; n_block = 1
PlotFourier_for_k_diff(N, n_hid, n_block, lmbdas)

In [ ]:
# for fixed k, plot Fourier work w.r.t λ
def PlotFourier_λ(N, n_hid, n_block, lmbdas):
    if not os.path.exists('_Figure/Work/Fourier/'): os.mkdir('_Figure/Work/Fourier/')
    if not os.path.exists('_Figure/Work/Fourier/N={0}/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/N={0}/Real/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/Real/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/N={0}/Imaginary/'.format(N)): os.mkdir('_Figure/Work/Fourier/N={0}/Imaginary/'.format(N))
    # load data
    summation_real = [0 for _ in range(len(lmbdas))]
    summation_imag = [0 for _ in range(len(lmbdas))]
    for k in range(N):
        reals = []; imags = [] # 길이가 lmbdas인 list
        real_std = []; imag_std = [] # 길이가 lmbdas인 list
        Data = ProcessFourierL(k,N,n_hid,n_block,lmbdas)
        for n in range(len(lmbdas)):
            # real value에 대한 mean
            reals.append((np.array(Data[n]).real).mean())
            real_std.append((np.array(Data[n]).real).std())
            # imaginary value에 대한 mean
            imags.append((np.array(Data[n]).imag).mean())
            imag_std.append((np.array(Data[n]).imag).std())
            # AW 전체에 대한 값
            summation_real[n] += np.array(Data[n]).real
            summation_imag[n] += np.array(Data[n]).imag

        # real part
        plt.figure(dpi=400)
        plt.errorbar(lmbdas, reals, real_std, capsize=3, fmt="o", ls="-", c='#1f77b4')
        plt.plot(np.array(lmbdas), AV.AW_k(k,N,τ,μ,K,Da,kbT,np.array(lmbdas)), c='black',marker='o',markersize=5)
        plt.axvline(x=-1.0,c='gray',ls=":")
        plt.axhline(y=+0.0,c='gray',ls=":")
        plt.grid()
        plt.xlabel('$\\lambda$'); plt.ylabel('$Re[\\hat{w}_k(\\lambda)]$')
        plt.savefig('_Figure/Work/Fourier/N={0}/Real/Long_k={1}.png'.format(N,k))
        plt.close()
        plt.clf()

        # imaginary part
        plt.figure(dpi=400)
        plt.errorbar(lmbdas, imags, imag_std, capsize=3, fmt="o", ls="-", c='#ff7f0e')
        plt.plot(np.array(lmbdas), np.zeros_like(np.array(lmbdas)), c='black', marker='o',markersize=5)
        plt.grid()
        plt.xlabel('$\\lambda$'); plt.ylabel('$Im[\\hat{w}_k(\\lambda)]$')
        plt.savefig('_Figure/Work/Fourier/N={0}/Imaginary/Long_k={1}.png'.format(N,k),bbox_inches='tight')
        plt.close()
        plt.clf()

    summation_real_mean = []
    summation_real_std = []
    summation_imag_mean = []
    summation_imag_std = []
    for n in range(len(lmbdas)):
        # length 2000인 numpy array
        summation_reals = summation_real[n]
        summation_imags = summation_imag[n]
        # append to mean and stds
        summation_real_mean.append(summation_reals.mean())
        summation_real_std.append(summation_reals.std())
        summation_imag_mean.append(summation_imags.mean())
        summation_imag_std.append(summation_imags.std())

    # 전체 real의 합의 fluctuation
    plt.figure(dpi=400)
    plt.grid()
    plt.errorbar(lmbdas, summation_real_mean, summation_real_std, capsize=3, fmt='o', ls="-", c='#1f77b4')
    plt.plot(np.linspace(-2.0,-0.5,100001), AV.AW(N,τ,μ,K,Da,kbT,np.linspace(-2.0,-0.5,100001)),c='black')
    plt.scatter(np.array(lmbdas),AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas)),c='black') 
    plt.axvline(x=-1.0,c='gray',ls=":")
    plt.axhline(y=+0.0,c='gray',ls=":")
    plt.xlabel("$\\lambda$")
    plt.savefig("_Figure/Work/Fourier/N={0}/Real/Long_Whole.png".format(N),bbox_inches='tight')
    plt.close();
    plt.clf();

    # 전체 imag합의 fluctuation
    plt.figure(dpi=400)
    plt.grid()
    plt.errorbar(lmbdas, summation_imag_mean, summation_imag_std, capsize=3, fmt='o', ls="-", c='#ff7f0e')
    plt.plot(np.linspace(-2.0,-0.5,100001), np.zeros_like(np.linspace(-2.0,-0.5,100001)),c='black')
    plt.scatter(np.array(lmbdas),np.zeros_like(np.array(lmbdas)),c='black') 
    plt.xlabel("$\\lambda$")
    plt.savefig("_Figure/Work/Fourier/N={0}/Imaginary/Long_Whole.png".format(N),bbox_inches='tight')
    plt.close();
    plt.clf();

Ns = [11]        
n_block = 1; n_hid = 5
for n in range(len(Ns)):
    PlotFourier_λ(Ns[n],n_hid,n_block,lmbdas)

<Figure size 640x480 with 0 Axes>

In [ ]:
def PlotFourier_λ_diff(N, n_hid, n_block, lmbdas):
    if not os.path.exists('_Figure/Work/Fourier/Diff/N={0}/Real/'.format(N)): os.mkdir('_Figure/Work/Fourier/Diff/N={0}/Real/'.format(N))
    if not os.path.exists('_Figure/Work/Fourier/Diff/N={0}/Imaginary/'.format(N)): os.mkdir('_Figure/Work/Fourier/Diff/N={0}/Imaginary/'.format(N))
    # load data
    summation_real = [0 for _ in range(len(lmbdas))]
    summation_imag = [0 for _ in range(len(lmbdas))]
    for k in range(N):
        reals = []; imags = [] # 길이가 lmbdas인 list
        real_std = []; imag_std = [] # 길이가 lmbdas인 list
        Data = ProcessFourierL(k,N,n_hid,n_block,lmbdas)
        for n in range(len(lmbdas)):
            # real value에 대한 mean
            reals.append(abs(np.array(Data[n]).real - AV.AW_k(k,N,τ,μ,K,Da,kbT,np.array(lmbdas))).mean())
            real_std.append(abs(np.array(Data[n]).real - AV.AW_k(k,N,τ,μ,K,Da,kbT,np.array(lmbdas))).std())
            # imaginary value에 대한 mean
            imags.append(abs(np.array(Data[n]).imag).mean())
            imag_std.append(abs(np.array(Data[n]).imag).std())
            # AW 전체에 대한 값
            summation_real[n] += np.array(Data[n]).real
            summation_imag[n] += np.array(Data[n]).imag
        # plot
        plt.figure(dpi=400)
        plt.errorbar(lmbdas, reals, real_std, capsize=3, fmt="o", ls="-", c='#1f77b4')
        plt.axhline(y=0,c='black')
        plt.axvline(x=-1.0,c='gray',ls=":")
        plt.axhline(y=+0.0,c='gray',ls=":")
        plt.grid()
        plt.xlabel('$\\lambda$'); plt.ylabel('$Re[\\hat{w}_k(\\lambda)]$')
        plt.savefig('_Figure/Work/Fourier/N={0}/Real/Long_k={1}.png'.format(N,k))
        plt.close()
        plt.clf()

        # plot imaginary part
        plt.figure(dpi=400)
        plt.errorbar(lmbdas, imags, imag_std, capsize=3, fmt="o", ls="-", c='#ff7f0e')
        plt.axhline(y=0,c='black')
        plt.grid()
        plt.xlabel('$\\lambda$'); plt.ylabel('$Im[\\hat{w}_k(\\lambda)]$')
        plt.savefig('_Figure/Work/Fourier/N={0}/Imaginary/Long_k={1}.png'.format(N,k),bbox_inches='tight')
        plt.close()
        plt.clf()

    summation_real_mean = []
    summation_real_std = []
    summation_imag_mean = []
    summation_imag_std = []
    for n in range(len(lmbdas)):
        # length 2000인 numpy array
        summation_reals = summation_real[n]
        summation_imags = summation_imag[n]
        # append to mean and stds
        summation_real_mean.append(abs(summation_reals - AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas))).mean())
        summation_real_std.append(abs(summation_reals - AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas))).std())
        summation_imag_mean.append(abs(summation_imags).mean())
        summation_imag_std.append(abs(summation_imags).std())

    # 전체 real의 합의 fluctuation
    plt.figure(dpi=400)
    plt.grid()
    plt.errorbar(lmbdas, summation_real_mean, summation_real_std, capsize=3, fmt='o', ls="-", c='#1f77b4')
    plt.axhline(y=0,c='black')
    plt.axvline(x=-1.0,c='gray',ls=":")
    plt.axhline(y=+0.0,c='gray',ls=":")
    plt.xlabel("$\\lambda$")
    plt.savefig("_Figure/Work/Fourier/N={0}/Real/Long_Whole.png".format(N),bbox_inches='tight')
    plt.close();
    plt.clf();

    # 전체 imag합의 fluctuation
    plt.figure(dpi=400)
    plt.grid()
    plt.errorbar(lmbdas, summation_imag_mean, summation_imag_std, capsize=3, fmt='o', ls="-", c='#ff7f0e')
    plt.axhline(y=0,c='black')
    plt.xlabel("$\\lambda$")
    plt.savefig("_Figure/Work/Fourier/N={0}/Imaginary/Long_Whole.png".format(N),bbox_inches='tight')
    plt.close();
    plt.clf();

Ns = [11]        
n_block = 1; n_hid = 5
for n in range(len(Ns)):
    PlotFourier_λ_diff(Ns[n],n_hid,n_block,lmbdas)

### Large deviation function

In [38]:
def LDF_analytic(N,tau,mu,K,Da,kbT):
    c = -1/(2*kbT)
    lmbda = np.linspace(c-1.001,c+1.001,1000000)
    AW = AV.AW(N,tau,mu,K,Da,kbT,lmbda); CGF = AV.CGF(N,tau,mu,K,Da,kbT,lmbda); LDF = lmbda*AW-CGF
    return AW,LDF

In [39]:
# 하나의 N에 대해서 LDF를 그림
def PlotLDF(N,n_hid,n_block,lmbdas,lmbdas2):
    # True
    AAW, ALDF = LDF_analytic(N,τ,μ,K,Da,kbT) # 선
    AAWD = AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas2)); ALDFD = np.array(lmbdas2)*AAWD-AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas2)) # 점
    plt.figure(dpi=400)
    # reference
    plt.plot(AAW,ALDF,c='black')
    plt.scatter(AAWD,ALDFD,c='black',marker='o')
    # process & plot
    # load data
    loss = ProcessTrain("SCGF",N,n_hid,n_block,lmbdas); work = ProcessTrain("WORK",N,n_hid,n_block,lmbdas)
    ## data process
    LDF = [0.0]; AW = [AV.AW(N,τ,μ,K,Da,kbT,0.0)]
    for n in range(len(lmbdas)):
        LDF.append(lmbdas[n]*np.array(work[n]).mean() - np.array(loss[n]).mean())
        AW.append(np.array(work[n]).mean())
    plt.plot(AW,LDF,marker="o",markersize=3,ls="-",c='blue')
    # 꾸미기
    # plt.legend(bbox_to_anchor=(1.,1.))
    plt.xlabel("w"); plt.ylabel("$I(w)$")
    plt.xlim(-1+int(min(AW)), int(max(AW))+1)
    plt.ylim(ymax=1+int(max(ALDF)), ymin=0.001)
    if not os.path.exists("_Figure/LDF/N={0}".format(N)): os.mkdir("_Figure/LDF/N={0}".format(N))
    plt.savefig("_Figure/LDF/N={0}/LDF.png".format(N),bbox_inches='tight')
    # log plot
    plt.yscale("log")
    plt.savefig("_Figure/LDF/N={0}/LDF_log.png".format(N),bbox_inches='tight')
    plt.close()
    plt.clf()

n_hid = 5; n_block = 1
Ns = [11]
for N in Ns:
    PlotLDF(N,n_hid,n_block,[-0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0],
           [0.0, -0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0])

<Figure size 640x480 with 0 Axes>

In [44]:
def PlotLDF_long(N,n_hid,n_block,lmbdas,lmbdas2):
    plt.figure(dpi=400)
    # True
    AAW, ALDF = LDF_analytic(N,τ,μ,K,Da,kbT) # 선
    AAWD = AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas2)); ALDFD = np.array(lmbdas2)*AAWD-AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas2)) # 점
    plt.plot(AAW,ALDF,c='black',marker='o',markersize=5)
    # Load long data
    LDF = [0.0]; AW = [AV.AW(N,τ,μ,K,Da,kbT,0.0)]
    for lmbda in lmbdas:
        work = pd.read_pickle("data/Equivariant_LWORK_N={0}_H={1}_B={2}_λ={3}.pkl".format(N,n_hid,n_block,lmbda))
        scgf = pd.read_pickle("data/Equivariant_LSCGF_N={0}_H={1}_B={2}_λ={3}.pkl".format(N,n_hid,n_block,lmbda))
        AW.append(np.array(work[n]).mean())
        LDF.append(lmbdas[n]*np.array(work[n]).mean() + np.array(scgf[n]).mean())
    plt.plot(AW,LDF,marker='o',markersize=5,ls="-",c='blue')
    # 꾸미기
    plt.xlabel("w"); plt.ylabel("$I(w)$")
    plt.xlim(-1+int(min(AW)), int(max(AW))+1)
    plt.ylim(ymax=1+int(max(ALDF)), ymin=0.001)
    if not os.path.exists("_Figure/LDF/N={0}".format(N)): os.mkdir("_Figure/LDF/N={0}".format(N))
    plt.savefig("_Figure/LDF/N={0}/Long_LDF.png".format(N),bbox_inches='tight')
    # log plot
    plt.yscale("log")
    plt.savefig("_Figure/LDF/N={0}/Long_LDF_log.png".format(N),bbox_inches='tight')
    plt.close()
    plt.clf()

n_hid = 5; n_block = 1
Ns = [11]
for N in Ns:
    PlotLDF_long(N,n_hid,n_block,[-0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0],
                 [0.0, -0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0])

<Figure size 640x480 with 0 Axes>

### Detailed Flcutuation theorem

In [41]:
def PlotDFT(N,n_hid,n_block,lmbdas):
    # True
    c = -1/(2*kbT); AWs = AV.AW(N,τ,μ,K,Da,kbT,np.linspace(-2.01,0.01,1000000))
    plt.figure(dpi=400)
    plt.plot(AWs,2*c*AWs,color='black',ls=":",label="(True) Slope = {0:.2f}".format(2*c))
    LDF = [0.0]; AW = [AV.AW(N,τ,μ,K,Da,kbT,0.0)]
    # load data
    aw = ProcessTrain("WORK",N,n_hid,n_block,lmbdas)
    cgf = ProcessTrain("SCGF",N,n_hid,n_block,lmbdas)
    # process
    for n in range(len(lmbdas)):
        AW.append(np.array(aw[n][-2000:]).mean())
        LDF.append(lmbdas[n]*np.array(aw[n][-2000:]).mean() - np.array(cgf[n][-2000:]).mean())
    # let's reverse it!
    LDF_flip = np.flip(LDF)
    # slope값 구하기
    fit = linregress(AW,LDF-LDF_flip)
    # plot graph
    plt.scatter(AW,LDF-LDF_flip,marker="o",c='blue',s=5)
    plt.plot(AW,fit.slope*np.array(AW)+fit.intercept,ls="--",c='blue',label="Slope = {0:.2f}, Intercept = {1:.2f}".format(fit.slope,fit.intercept))
    plt.xlabel("$w$"); plt.ylabel("I(w) - I(-w)")
    plt.legend(loc='upper right')
    plt.grid()
    plt.savefig("_Figure/LDF/N={0}/DFT.png".format(N),bbox_inches='tight')
    plt.close()
    plt.clf()

n_hid = 5; n_block = 1
Ns = [11]
for N in Ns:
    PlotDFT(N,n_hid,n_block,lmbdas)

<Figure size 640x480 with 0 Axes>

In [48]:
def PlotDFT_all(Ns, n_hids, n_block, lmbdas, lrp, lrd):
    cs = [
    '#1f77b4',  # blue
    '#ff7f0e',  # orange
    '#2ca02c',  # green
    '#d62728',  # red
    '#9467bd',  # purple
    '#8c564b',  # brown
    '#e377c2',  # pink
    '#7f7f7f',  # gray
    '#bcbd22',  # yellow-green
    '#17becf'   # cyan
    ]
    # True
    c = -1/(2*kbT); AWs = AV.AW(max(Ns),τ,μ,K,Da,kbT,np.linspace(-2.01,0.01,1000000))
    plt.figure(dpi=400)
    plt.plot(AWs,2*c*AWs,color='black',ls=":",label="(true) slope = {0:.2f}".format(2*c))
    for n in range(len(Ns)):
        LDF = [0.0]; AW = [AV.AW(Ns[n],τ,μ,K,Da,kbT,0.0)]
        # load data
        aw = ProcessTrain("WORK",Ns[n],n_hids[n],n_block,lmbdas,lrp,lrd)
        cgf = ProcessTrain("SCGF",Ns[n],n_hids[n],n_block,lmbdas,lrp,lrd)
        # process
        for m in range(len(lmbdas)):
            AW.append(np.array(aw[m][-2000:]).mean())
            LDF.append(lmbdas[m]*np.array(aw[m][-2000:]).mean() - np.array(cgf[m][-2000:]).mean())
        # let's reverse it!
        LDF_flip = np.flip(LDF)
        # slope
        fit = linregress(AW, LDF-LDF_flip)
        # plot
        plt.scatter(AW, LDF-LDF_flip, marker="o", s=5)
        plt.plot(AW, fit.slope*np.array(AW)+fit.intercept, c=cs[n], ls="--", label="N = {0}, slope = {1:.2f}".format(Ns[n], fit.slope))
    plt.xlabel("$w$"); plt.ylabel("I(w) - I(-w)")
    plt.grid()
    plt.legend(loc='upper right')
    plt.savefig("_Figure/LDF/ALLDFT.png".format(N),bbox_inches='tight')
    plt.close()
    plt.clf()
PlotDFT_all([7, 9, 11], [5,5,5], 0, lmbdas, 0.1, 1e-5)

<Figure size 640x480 with 0 Axes>